# 文本信息预处理

一篇文章可以被简单地看作一串单词序列，甚至是一串字符序列。 本节中，我们将解析文本的常见预处理步骤。 这些步骤通常包括：
* 将文本作为字符串加载到内存中。
* 将字符串拆分为词元（如单词和字符）。
* 建立一个词表，将拆分的词元映射到数字索引。
* 将文本转换为数字索引序列，方便模型操作。

## 读取数据集

In [5]:
import collections
import re
from d2l import torch as d2l

d2l.DATA_HUB['time_machine'] = (d2l.DATA_URL + 'timemachine.txt',
                                '090b5e7e70c295757f55df93cb0a180b9691891a')

def read_time_machine():  
    """将时间机器数据集加载到文本行的列表中"""
    with open(d2l.download('time_machine'), 'r') as f:
        lines = f.readlines()
    return [re.sub('[^A-Za-z]+', ' ', line).strip().lower() for line in lines]  # 依次为移除非字母字符、去除首尾空白字符，转换为小写

lines = read_time_machine()
print(f'# 文本总行数: {len(lines)}')
for i in range(11):
    print(lines[i])

# 文本总行数: 3221
the time machine by h g wells




i


the time traveller for so it will be convenient to speak of him
was expounding a recondite matter to us his grey eyes shone and
twinkled and his usually pale face was flushed and animated the


## 词元化

下面的tokenize函数将文本行列表（lines）作为输入， 列表中的每个元素是一个文本序列（如一条文本行）。 每个文本序列又被拆分成一个词元列表，词元（token）是文本的基本单位。 最后，返回一个由词元列表组成的列表，其中的每个词元都是一个字符串（string）。

In [ ]:
def tokenize(lines, token='word'):  #@save
    """将文本行拆分为单词或字符词元"""
    if token == 'word':
        return [line.split() for line in lines]  # 如果是按照词元拆分
    elif token == 'char':
        return [list(line) for line in lines]  # 按照字符拆分，返回一个列表的列表
    else:
        print('错误：未知词元类型：' + token)

tokens = tokenize(lines)  # lines是一个list of list
for i in range(11):
    print(tokens[i])

['the', 'time', 'machine', 'by', 'h', 'g', 'wells']
[]
[]
[]
[]
['i']
[]
[]
['the', 'time', 'traveller', 'for', 'so', 'it', 'will', 'be', 'convenient', 'to', 'speak', 'of', 'him']
['was', 'expounding', 'a', 'recondite', 'matter', 'to', 'us', 'his', 'grey', 'eyes', 'shone', 'and']
['twinkled', 'and', 'his', 'usually', 'pale', 'face', 'was', 'flushed', 'and', 'animated', 'the']


## 构建词表

词元的类型是字符串，而模型需要的输入是数字，因此这种类型不方便模型使用。 现在，让我们构建一个字典，通常也叫做词表（vocabulary）， 用来将字符串类型的词元映射到从0开始的数字索引中。我们先将训练集中的所有文档合并在一起，对它们的唯一词元进行统计， 得到的统计结果称之为语料（corpus）。 然后根据每个唯一词元的出现频率，为其分配一个数字索引。

 另外，语料库中不存在或已删除的任何词元都将映射到一个特定的未知词元“\<unk>”。 我们可以选择增加一个列表，用于保存那些被保留的词元， 例如：填充词元（“\<pad>”）； 序列开始词元（“\<bos>”）； 序列结束词元（“\<eos>”）。

In [ ]:
def count_corpus(tokens):  # 统计词元的频率
    """统计词元的频率"""
    # 这里的tokens是1D列表或2D列表
    if len(tokens) == 0 or isinstance(tokens[0], list):  # isinstance(tokens[0], list)：这是一个类型判断函数。意思是“判断 tokens 的第一个元素是不是一个列表”。
        # 将词元列表展平成一个列表
        tokens = [token for line in tokens for token in line]  # 先找在tokens里面找line， 再在line里面找token，放进tokens列表
# 上面的if分支是要展平整个词元
    return collections.Counter(tokens)   # 统计tokens列表中每个独一无二的词元出现的次数

class Vocab:  #@save
    """文本词表"""
    def __init__(self, tokens=None, min_freq=0, reserved_tokens=None):  # reserved_tokens是保留词元列表，比如未知词元“\<unk>”等
        if tokens is None:
            tokens = []
        if reserved_tokens is None:
            reserved_tokens = []
        # 按出现频率排序
        counter = count_corpus(tokens)  # tokens是1D列表或2D列表，返回一个字典，键是词元，值是词元出现的次数
        self._token_freqs = sorted(counter.items(), key=lambda x: x[1],
                                   reverse=True)  # 按照词元出现的次数从高到低排序
        # idx_to_token维护的是索引到词元的映射关系
        self.idx_to_token = ['<unk>'] + reserved_tokens  # 这里表示列表的拼接，这里表示将未知词元“\<unk>”添加到保留词元列表的开头
        self.token_to_idx = {token: idx
                             for idx, token in enumerate(self.idx_to_token)}
        # 这里表示的是词元到idx的映射，维护的是一个字典，key是词元，value对应的是词元的索引
        for token, freq in self._token_freqs:  # self._token_freqs维护的是一个按出现次数降序排列的token有序字典
            if freq < min_freq:  #低于最小频次的token不再考虑
                break  # 这里是建立在降序排列的情况下的
            if token not in self.token_to_idx:
                self.idx_to_token.append(token)  # idx_to_token列表的长度就是词表的大小
                self.token_to_idx[token] = len(self.idx_to_token) - 1 #映射新的词元到索引，减1是为了对应idx_to_token的下标
# token_to_idx维护的是词元到索引的映射关系
    def __len__(self):
        return len(self.idx_to_token)

    def __getitem__(self, tokens):  # 魔术方法，用于实现索引操作，比如 vocab['hello']，这就等同于调用了 __getitem__('hello')
# 该函数可以让对象像字典一样被查询
        if not isinstance(tokens, (list, tuple)):    # 单个字符串直接返回
            return self.token_to_idx.get(tokens, self.unk)  # 字典的get方法，如果tokens不在字典中，返回self.unk，否则返回tokens对应的值
        return [self.__getitem__(token) for token in tokens]  # 如果tokens是列表或元组，递归调用__getitem__方法，返回一个列表
# 这里是做递归调用
# 递归终止条件：当输入 tokens 不是列表或元组时（即单个词元），直接返回该词元对应的索引
# 递归调用：当输入 tokens 是列表或元组时，对每个元素递归调用 __getitem__
    def to_tokens(self, indices):  # 数字转换为词元
        if not isinstance(indices, (list, tuple)):  # 如果不是列表和元组，可以直接进行查询
            return self.idx_to_token[indices]
        return [self.idx_to_token[index] for index in indices]

    @property  # 装饰器，可以将一个方法转换为只读属性，访问方式就像访问实例变量一样。
    def unk(self):  # 未知词元的索引为0
        return 0

    @property
    def token_freqs(self):
        return self._token_freqs

打印前几个高频词元和索引

In [ ]:
vocab = Vocab(tokens)
print(list(vocab.token_to_idx.items())[:10])  # 这里输出的是索引的编号，而不是对应的频率

[('<unk>', 0), ('the', 1), ('i', 2), ('and', 3), ('of', 4), ('a', 5), ('to', 6), ('was', 7), ('in', 8), ('that', 9)]


## 整合功能

该函数返回corpus（词元索引列表）和vocab（时光机器语料库的词表）。 我们在这里所做的改变是：

* 为了简化后面章节中的训练，我们使用字符（而不是单词）实现文本词元化；

* 时光机器数据集中的每个文本行不一定是一个句子或一个段落，还可能是一个单词，因此返回的corpus仅处理为单个列表，而不是使用多词元列表构成的一个列表。

In [ ]:
def load_corpus_time_machine(max_tokens=-1):  #@save
    """返回时光机器数据集的词元索引列表和词表"""
    lines = read_time_machine()
    tokens = tokenize(lines, 'char')  # 这里切出来是嵌套的列表，每一个嵌套的列表表示一行的字符
    vocab = Vocab(tokens)
    # 因为时光机器数据集中的每个文本行不一定是一个句子或一个段落，
    # 所以将所有文本行展平到一个列表中
    corpus = [vocab[token] for line in tokens for token in line]  # 这里将每个字符转换为对应的索引，
# 这个其实就是返回了所有字符对应的index
    if max_tokens > 0:  # 设置的最大的词元数量，超过这个数量的词元会被截断（只取次数在前max_tokens的高频词元索引）
        corpus = corpus[:max_tokens]
    return corpus, vocab  # corpus返回的高频词元的idx， vocab是词表

corpus, vocab = load_corpus_time_machine()
len(corpus), len(vocab)  # 这里vcob是调用了内置的魔术方法

(170580, 28)